# SECOND LAB

### Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import *
from IPython.display import Markdown, display
from huggingface_hub import login
from tqdm.notebook import tqdm
from openai import OpenAI
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import argparse
import requests
import torch
import re
import json
from bs4 import BeautifulSoup

In [2]:
# Check for GPU / CPU
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = "cuda" if torch.cuda.is_available() else "cpu"

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA: 12.1


### Connecting to Hugging Face

The next cell is where we load in the environment variables in your `.env` file and connect to Hugging Face.  

In [3]:
# Log in to HuggingFace - if you get a "Note" about Environment variable being set, ignore it

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Constants

In [4]:
# Constants
website_id = "https://karanoberoi28.github.io/"

# Model details:
# model_id = "google/gemma-3-4b-it"
model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Inputs
tone = "professional"
language = "English"


### Fetch Website Content

In [5]:
# Extract website content

def extract_website(url):

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        # Raise an exception for HTTP error codes (4xx or 5xx)
        response.raise_for_status() 

        # Parse HTML content
        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unnecessary elements
        for element in soup([
            "script", "style", "noscript", "iframe",
            "svg", "header", "footer", "nav", "aside"
        ]):
            element.decompose()

        # Extract page title
        title = soup.title.get_text(strip=True) if soup.title else ""

        # Page content
        content = (
            soup.find("article")
            or soup.find("main")
            or soup.find("body")
            or soup
        )

        # Extract readable text
        text = content.get_text(separator="\n", strip=True)

        # Remove empty lines and duplicate whitespace
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        text = "\n".join(lines)

        return {
            "url": url,
            "title": title,
            "content": text
        }

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching the website: {e}")
        return None

    except Exception as e:
        print(f"An error occurred while extracting content: {e}")
        return None

### Prompt Definition

In [6]:
# Prompt
PAMPHLET_PROMPT = f"""You are a talented copywriter who designs concise, persuasive pamphlets post analyzing the contents of several relevant pages from a company website
and creates a short brochure about the company or personal website. Respond in markdown without code blocks.

Using the WEBSITE below, write a pamphlet in Markdown.

Style: {tone}
Language: {language}

Required layout:
# <Title>
> <one-line tagline>

## About Us / About This
(2-3 short sentences as website summary)

## Get In Touch / Next Steps
(clear call to action + contact details that were provided)

Hard rules:
- Do NOT invent facts, prices, dates, testimonials or contact details.
- Keep it scannable: short paragraphs, strong verbs, fits on one printed A4/folded page.
- Output ONLY the pamphlet Markdown.

WEBSITE:
\"\"\"
{extract_website(website_id)['content']}
\"\"\"
"""

### Generate Pamphlet

In [7]:
# Load model directly
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [12]:
# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load model directly onto available hardware
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map=device,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)

print("Model loaded successfully!")
print("Memory footprint:", round(model.get_memory_footprint() / 1024**3, 2), "GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Memory footprint: 2.05 GB


In [13]:
# Defining prompt & message
messages=[
        {"role": "user", "content": PAMPHLET_PROMPT}
    ]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(device)

In [16]:
# Output
outputs = model.generate(**inputs, max_new_tokens=250)

# Extract Response
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:], 
    clean_up_tokenization_spaces=False)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [17]:
# Extract Output
print(response)

# Solving Complex Problems
Unlocking the Power of Data Science and AI

## About Us / About This
Karan, a seasoned data scientist and AI expert, leads a multidisciplinary team in developing production-grade products that combine the capabilities of Generative AI with traditional analytical and physics-based models. With over 16 years of experience, Karan has worked across industries, including Aviation, Life Sciences, and Supply Chain, delivering significant value to organizations.

## Get In Touch / Next Steps
Get in touch with Karan to explore how his team can solve complex problems for your organization. Contact:
Email: [Not provided]
Phone: [Not provided]
Let's unlock the power of data science and AI together.<|eot_id|>


In [18]:
# Clear GPU memory and model
del outputs
del inputs
del model

torch.cuda.empty_cache()
torch.cuda.ipc_collect()